<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Install Libraries
</h2>

In [1]:
!pip install langchain-community faiss-cpu langchain-core groq jina pymupdf tabula-py requests pillow transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.9/378.9 kB 11.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of types-requests to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of types-requests to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Importing the libs
</h2>

In [2]:
import faiss
import json
import base64
import pymupdf
import requests
import os
import logging
import numpy as np
import warnings
from tqdm import tqdm
from typing import List, Dict, Any

from PIL import Image
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer
from groq import Groq

logger = logging.getLogger(__name__)
logger.setLevel(logging.ERROR)

warnings.filterwarnings("ignore")

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Set Up Credentials & Models
</h2>

In [3]:
GROQ_API_KEY = "GROQ_API_KEY"

if GROQ_API_KEY is None:
    print("Warning: GROQ_API_KEY environment variable not set. Please set it to use the Groq LLM.")
else:
    groq_client = Groq(api_key=GROQ_API_KEY)

In [4]:
# Load HuggingFace CLIP model and processor
print("Loading CLIP model...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Loading Jina v4 embedding model for text/tables/images...")
#embedding_model_id = "jinaai/jina-embeddings-v4-vllm-retrieval"
#embed_model = SentenceTransformer(embedding_model_id, trust_remote_code=True)
embedding_model_id = "abhinand/MedEmbed-large-v0.1"
embed_model = SentenceTransformer(embedding_model_id)

Loading CLIP model...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Embedding Functions
</h2>

In [5]:
def get_text_embedding(text: str) -> np.ndarray:
    """Generates embeddings for the given text using Jina AI model."""
    try:
        emb = embed_model.encode([text], batch_size=64,
                                        convert_to_numpy=True, normalize_embeddings=True)[0]
        #emb = embed_model.encode([text], convert_to_numpy=True, show_progress_bar=False, task='retrieval')[0]
        return emb.astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating text/table embedding: {e}")
        return None

def get_image_embedding(image_path: str) -> np.ndarray:
    """Generates embeddings for an image using HuggingFace CLIP's image encoder."""
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors='pt')
        image_features = clip_model.get_image_features(**inputs)
        return image_features.detach().numpy().flatten().astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating image embedding for {image_path}: {e}")
        return None
def get_clip_text_embedding(text: str) -> np.ndarray:
    """Generates embeddings for text using HuggingFace CLIP's text encoder."""
    try:
        inputs = clip_processor(text=text, return_tensors='pt', padding=True, truncation=True)
        text_features = clip_model.get_text_features(**inputs)
        return text_features.detach().numpy().flatten().astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating CLIP text embedding: {e}")
        return None


In [6]:
#CLIP Text Embedder
def get_clip_text_embedding(text: str) -> np.ndarray:
    """Generates embeddings for text using HuggingFace CLIP's text encoder."""
    try:
        inputs = clip_processor(text=text, return_tensors='pt', padding=True, truncation=True)
        text_features = clip_model.get_text_features(**inputs)
        return text_features.detach().numpy().flatten().astype(np.float32)
    except Exception as e:
        logger.error(f"Error generating CLIP text embedding: {e}")
        return None

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Data Loading
</h2>

In [7]:
import os

# Set the filename and filepath (already in data/)
filename = "Prot_000.pdf"
filepath = os.path.join(filename)

# Make sure the file exists
if not os.path.exists(filepath):
    raise FileNotFoundError(f"{filepath} not found! Please upload it to the data/ folder.")
else:
    print(f"Using file: {filepath}")


Using file: Prot_000.pdf


<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Data Extraction
</h2>

In [8]:
def create_directories(base_dir):
    directories = ["images", "text", "tables"]
    for dir in directories:
        os.makedirs(os.path.join(base_dir, dir), exist_ok=True)

def process_tables(doc, page_num, base_dir, items, filepath):
    try:
        import tabula
        tables = tabula.read_pdf(filepath, pages=page_num + 1, multiple_tables=True)
        if not tables:
            return
        for table_idx, table in enumerate(tables):
            table_text = "\n".join([" | ".join(map(str, row)) for row in table.values])
            table_file_name = f"{base_dir}/tables/{os.path.basename(filepath)}_table_{page_num}_{table_idx}.txt"
            with open(table_file_name, 'w') as f:
                f.write(table_text)
            items.append({"page": page_num, "type": "table", "text": table_text, "path": table_file_name})
    except Exception as e:
        print(f"Error extracting tables from page {page_num}: {str(e)}")

def process_text_chunks(text, page_num, base_dir, items, filepath):
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=200)
    chunks = text_splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        text_file_name = f"{base_dir}/text/{os.path.basename(filepath)}_text_{page_num}_{i}.txt"
        with open(text_file_name, 'w') as f:
            f.write(chunk)
        items.append({"page": page_num, "type": "text", "text": chunk, "path": text_file_name})

def process_images(doc, page_num, base_dir, items, filepath):
    page = doc[page_num]
    images = page.get_images()
    for idx, image in enumerate(images):
        xref = image[0]
        try:
            pix = pymupdf.Pixmap(doc, xref)
            if pix.colorspace.n > 3:  # Check if colorspace is not RGB or Grayscale
                pix = pymupdf.Pixmap(pymupdf.csRGB, pix) # Convert to RGB

            image_name = f"{base_dir}/images/{os.path.basename(filepath)}_image_{page_num}_{idx}_{xref}.png"
            pix.save(image_name)
            items.append({"page": page_num, "type": "image", "path": image_name})
        except Exception as e:
            print(f"Error processing image {idx} on page {page_num}: {e}")


def extract_items_from_pdf(filepath):
    doc = pymupdf.open(filepath)
    num_pages = len(doc)
    base_dir = "data"
    create_directories(base_dir)
    items = []

    for page_num in tqdm(range(num_pages), desc="Processing PDF pages"):
        page = doc[page_num]
        text = page.get_text()
        process_tables(doc, page_num, base_dir, items, filepath)
        process_text_chunks(text, page_num, base_dir, items, filepath)
        process_images(doc, page_num, base_dir, items, filepath)
    return items

In [9]:
items = extract_items_from_pdf(filepath)

Processing PDF pages:   0%|          | 1/233 [00:01<06:16,  1.62s/it]WARNING:tabula.backend:Got stderr: Aug 28, 2025 9:00:04 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 9:00:04 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 9:00:05 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 9:00:05 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile

Processing PDF pages:   1%|          | 2/233 [00:03<06:00,  1.56s/it]WARNING:tabula.backend:Got stderr: Aug 28, 2025 9:00:05 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 9:00:06 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 9:00:06 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 9:00:06 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile

Processing PDF

In [10]:
# Looking at the first text item
[i for i in items if i['type'] == 'text'][0]

{'page': 0,
 'type': 'text',
 'text': 'Official Title of Study: \nA Phase 1/2 Study of the Combination of Lirilumab (Anti-KIR) Plus Nivolumab (Anti-PD-1) or Lirilumab \nPlus Nivolumab and Ipilimumab in Advanced Refractory Solid Tumors \nNCT Number: NCT01714739 \nDocument Date (Date in which document was last revised): May 8, 2018',
 'path': 'data/text/Prot_000.pdf_text_0_0.txt'}

In [11]:
# Looking at the first table item
[i for i in items if i['type'] == 'table'][0]

{'page': 3,
 'type': 'table',
 'text': 'Document | Date of Issue | Summary of Change\nnan | nan | The primary purpose of this revised protocol is to close the future\nnan | nan | enrollment in Part 3 and Part 5 and removal of Part 4 and Part 6 from the\nnan | nan | protocol study design. Additional revisions based on the lack of clear\nRevised | nan | nan\nnan | 08-May-2018 | evidence of clinical benefit will include removal of overall survival visits\nProtocol 12 | nan | nan\nnan | nan | for all subjects, additional subjects entering treatment beyond\nnan | nan | progression, retreatment at the time of disease progression as well as\nnan | nan | select sample collection and study visits.\nAdministrative | nan | nan\nnan | 30-Aug-2017 | Medical Monitor update\nLetter 05 | nan | nan\nAdministrative | nan | nan\nnan | 30-May-2017 | Medical Monitor and Study Director update\nLetter 04 | nan | nan\nRevised | nan | nan\nnan | 28-Feb-2017 | Incorporates Amendment 15\nProtocol 11 | nan | nan\

In [12]:
# Looking at the first image item
[i for i in items if i['type'] == 'image'][0]

{'page': 18,
 'type': 'image',
 'path': 'data/images/Prot_000.pdf_image_18_0_205.png'}

<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   Vector Search and Retrieval
</h2>

In [16]:
# Generate embeddings and prepare documents for vector stores
text_embeddings = []
text_docs = []
image_embeddings = []
image_docs = []

print("Generating embeddings for text and table items...")
for item in tqdm(items, desc="Processing items"):
    if item['type'] in ['text', 'table']:
        embedding = get_text_embedding(item['text'])
        if embedding is not None:
            text_embeddings.append(embedding)
            text_docs.append(item)
    elif item['type'] == 'image':
        embedding = get_image_embedding(item['path'])
        if embedding is not None:
            image_embeddings.append(embedding)
            image_docs.append(item)

# Create FAISS indices
print("Creating text and image FAISS indices...")
text_embedding_dim = text_embeddings[0].shape[0]
text_index = faiss.IndexFlatL2(text_embedding_dim)
text_index.add(np.array(text_embeddings))

image_embedding_dim = image_embeddings[0].shape[0]
image_index = faiss.IndexFlatL2(image_embedding_dim)
image_index.add(np.array(image_embeddings))

Generating embeddings for text and table items...


Processing items: 100%|██████████| 1168/1168 [00:27<00:00, 42.01it/s]


Creating text and image FAISS indices...


<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
   RAG Pipeline with Groq
</h2>

In [29]:
def invoke_groq_multimodal(query: str, retrieved_items: List[Dict[str, Any]]) -> str:
    """Generates a response from the retrieved context using Groq LLM."""
    context = ""

    for item in retrieved_items:
        if item['type'] in ['text', 'table']:
            context += f"\n\nText from page {item['page']}:\n{item['text']}"
        elif item['type'] == 'image':
            context += f"\n\nImage from page {item['page']}:\nPath: {item['path']}"

    prompt_template = f"""You are a helpful assistant. Use the following pieces of text and image information to answer the user's question. If the answer is not in the provided context, politely say that you cannot provide an answer. Do not use any external knowledge. If an image is relevant, mention its file path. \n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""

    messages = [
        {"role": "user", "content": prompt_template}
    ]

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=2048,
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error invoking Groq: {str(e)}"

def rag_pipeline(query: str):
    # Get embeddings for the query
    text_query_embedding = get_text_embedding(query)

    # The image search must use the CLIP text encoder to embed the query
    image_query_embedding = get_clip_text_embedding(query)

    retrieved_items = []

    # A specific check for the user's direct command to list all tables
    if query.lower().strip() == "show all the tables in this file":
        retrieved_items = [item for item in items if item['type'] == 'table']

    # If not a specific command, proceed with the normal RAG pipeline
    else:
        if text_query_embedding is not None:
            # Search text index
            text_distances, text_results = text_index.search(np.array([text_query_embedding]), k=3)
            for idx in text_results.flatten():
                retrieved_items.append(text_docs[idx])

        if image_query_embedding is not None:
            # Search image index
            image_distances, image_results = image_index.search(np.array([image_query_embedding]), k=2)
            for idx in image_results.flatten():
                retrieved_items.append(image_docs[idx])

    # Pass combined context to LLM
    if retrieved_items:
        response = invoke_groq_multimodal(query, retrieved_items)
        return response
    else:
        return "No relevant items found."

print(rag_pipeline("what are the Prohibited and/or Restricted Treatments"))

The Prohibited and/or Restricted Treatments are listed in section 3.4.1 and include:

* Immunosuppressive agents (except as stated in Section 3.4.3)
* Immunosuppressive doses of systemic corticosteroids (except as stated in Sections 3.4.2 and 3.4.3)
* Any concurrent anti-neoplastic therapy (i.e., chemotherapy, hormonal therapy, immunotherapy, extensive, non-palliative radiation therapy, or standard or investigational agents)

This information can be found on page 99 of the text. The images from page 72 (data/images/Prot_000.pdf_image_72_0_1015.png and data/images/Prot_000.pdf_image_72_1_1016.png) do not appear to be relevant to this specific question.


<h2 style="background: linear-gradient(to right, #ff6b6b, #4ecdc4, #1e90ff);
            color: white;
            padding: 15px;
            border-radius: 10px;
            text-align: center;
            font-family: 'Comic Sans MS', cursive, sans-serif;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Thank you!
</h2>

In [12]:
items = extract_items_from_pdf(filepath)

Processing PDF pages:   0%|          | 1/233 [00:01<03:54,  1.01s/it]WARNING:tabula.backend:Got stderr: Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:54 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile

Processing PDF pages:   1%|          | 2/233 [00:02<04:46,  1.24s/it]WARNING:tabula.backend:Got stderr: Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile
Aug 28, 2025 8:31:55 PM org.apache.pdfbox.pdmodel.graphics.color.PDICCBased ensureDisplayProfile

Processing PDF

In [30]:
import os

base_dir = "data"
subfolders = ["images", "tables", "text"]

for sub in subfolders:
    folder_path = os.path.join(base_dir, sub)
    if os.path.exists(folder_path):
        files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
        print(f"{sub}: {len(files)} files")
    else:
        print(f"{sub}: folder not found")


images: 19 files
tables: 93 files
text: 1056 files
